# Anomaly detection of a thyratron grid waveform

Copyright (C) 2024 Hirokazu Maesaka, RIKEN SPring-8 Center. All rights reserved.

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import os
import subprocess
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
plt.rcParams['font.size'] = 16
%matplotlib inline

## Mount google drive to get data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
basedir='/content/drive/MyDrive/Colab Notebooks/ISBA24'

# Checking (preparing) data files

In [ ]:
if not os.path.isdir(basedir+'/data'):
  print('Data directory not found, trying to decompress data archive.')
  if not os.path.isfile(f'{basedir}/wfmdata.tar.gz'):
    print('Data archive not found.')
    raise FileNotFoundError
  print(subprocess.check_output(['tar', 'zxvf', basedir+'/wfmdata.tar.gz', '-C', basedir+'/']).decode())
datadir=basedir+'/data'

# Classification of input data

In [ ]:
thyIdList=[[],[],[]] # new, aged, failed
dfList=[[],[],[]]
thyData=glob.glob(datadir+'/sn*_g1.npz')
for data in thyData:
  dirname=os.path.basename(data)
  serial, hrs=re.findall('sn(\d+)_(\d+)h_', dirname)[0]
  serial=int(serial)
  hrs=int(hrs)

  tIdx=-1
  if hrs<5000: # 1500
    tIdx=0
  elif hrs<20000:
    tIdx=1
  else:
    tIdx=2

  tdf=pd.DataFrame(np.load('%s/sn%d_%dh_g1.npz'%(datadir, serial, hrs))['wfm'])
  thyIdList[tIdx].append([serial, hrs])
  dfList[tIdx].append(tdf)

print('new: %d'%len(thyIdList[0]))
print('aged: %d'%len(thyIdList[1]))
print('failed: %d'%len(thyIdList[2]))

# S/N: 2176 (18100hr) seems to be in good condition

## Training data

In [ ]:
tdflist=[]
for i, dfl in enumerate(dfList):
  if i==1: # aged
    continue

  tdflist.append(pd.concat(dfl))
tdf=pd.concat(tdflist)

# PCA

In [ ]:
pca=PCA(n_components=6)
pca.fit(tdf)

## Cumulative contribution ratio

In [ ]:
cr=pd.DataFrame(np.insert(pca.explained_variance_ratio_,0,0))
plt.plot(cr.cumsum(),'o-')
plt.ylim(0.,1.05)
plt.grid()
plt.xlabel('Number of principal components')
plt.ylabel('Cumulative contribution ratio')
plt.show()

## Plot all the PCA outputs

In [ ]:
fig=plt.figure(figsize=[8,8])

pcadf=None
mk = ['+','x','.']
for i, dfl in enumerate(dfList):
  for j, df in enumerate(dfl[:10]):
    ptf=pca.transform(df)
    plt.scatter(ptf[:,0], ptf[:,1], marker=mk[i], label='SN: %d (%dhr)'%(thyIdList[i][j][0],thyIdList[i][j][1]))

plt.xlabel('PCA-1')
plt.ylabel('PCA-2')

plt.grid()
plt.legend(loc='center left', bbox_to_anchor=(1., 0.5))
plt.show()

## Plot PCA outputs for training data

In [ ]:
fig=plt.figure(figsize=[8,8])

pcadf=None
for i, dfl in enumerate(dfList):
  if i==1: # aged
    continue

  for j, df in enumerate(dfl[:10]):
    ptf=pca.transform(df)
    plt.scatter(ptf[:,0], ptf[:,1], marker='+' if i==0 else '.', label='SN: %d (%dhr)'%(thyIdList[i][j][0],thyIdList[i][j][1]))

    if i==2 and thyIdList[i][j][0]==2010:
      pcadf=pd.DataFrame(ptf[:,:2])

plt.xlabel('PCA-1')
plt.ylabel('PCA-2')

plt.grid()
plt.legend(loc='center left', bbox_to_anchor=(1., 0.5))
plt.show()

## Waveform of each component

In [ ]:
for i in range(pca.components_.shape[0]):
    fig, ax = plt.subplots()
    ax.plot(pca.components_[i,:])
    ax.grid()
    ax.set_title(f'Component {i+1}')
    plt.show()

## Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Select new and failed tubes
xtrainlist = []
ytrainlist = []
xtestlist = []
ytestlist = []
for i, dfl in enumerate(dfList):
    for j, df in enumerate(dfl):
        ptf=pca.transform(df)
        if i==1:
            xtestlist.append(ptf)
            ytestlist.append(np.zeros(ptf.shape[0]))
        else:
            xtrainlist.append(ptf)
            ytrainlist.append(np.ones(ptf.shape[0])*(i//2))
xtrain = np.vstack(xtrainlist)
ytrain = np.hstack(ytrainlist)
xtest = np.vstack(xtestlist)
ytest = np.hstack(ytestlist)
print(xtrain.shape, ytrain.shape)
print(xtest.shape, ytest.shape)

In [ ]:
# Fit
clf = LinearRegression().fit(xtrain,ytrain)

In [ ]:
fig, ax = plt.subplots()
ax.hist(clf.predict(xtrain[ytrain==0]), 100, [-0.5,1.5], alpha=0.8, label='train new')
ax.hist(clf.predict(xtrain[ytrain==1]), 100, [-0.5,1.5], alpha=0.8, label='train failed')
ax.hist(clf.predict(xtest), 100, [-0.5,1.5], alpha=0.8, label='test aged')
ax.grid()
ax.legend()
ax.set_title('Linear regression')
plt.show()

## Support vector machine (SVM)

In [ ]:
from sklearn import svm

In [ ]:
# Support vector machine regressor
clf = svm.SVR()
clf.fit(xtrain, ytrain)

In [ ]:
fig, ax = plt.subplots()
ax.hist(clf.predict(xtrain[ytrain==0]), 100, [-0.5,1.5], alpha=0.8, label='train new')
ax.hist(clf.predict(xtrain[ytrain==1]), 100, [-0.5,1.5], alpha=0.8, label='train failed')
ax.hist(clf.predict(xtest), 100, [-0.5,1.5], alpha=0.8, label='test aged')
ax.grid()
ax.legend()
ax.set_title('SVM regression')
plt.show()